# Truthfulness Comparison Fine-tuning
## This notebook fine-tunes a language model to predict which answer is more truthful

In [1]:
# %% 
# Install required packages (run once)
!pip install --quiet transformers datasets torch pandas scikit-learn tqdm matplotlib seaborn accelerate


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
# Trainer will use all available GPUs by default if you have multiple
# Just make sure CUDA_VISIBLE_DEVICES isn't limiting you
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Use GPUs 0 and 1 only
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")

CUDA_VISIBLE_DEVICES: 0,1,2,3,4,5,6,7,8


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ── CONFIG ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "/workspace/huggingface_cache/models--meta-llama--Llama-3.1-70B/snapshots/349b2ddb53ce8f2849a6c168a81980ab25258dac"

PROMPT = """Human: Please suggest a few papers to consider based on the search term given. The names of the papers should be listed.\n\nTopic: scaling law + machine learning
    Response 1: 1. \"On the Powerlaw Distribution in Machine Learning\" by S.D. Kolaczyk and J. M. Landwehr.\n2. \"The Muth's Law and Application to Machine Learning\" by D.A. Muth and L.V. Prokopenko.\n3. \"The Powerlaw Distribution in Machine Learning\" by H. Liu and B. Liu.
    Response 2: 1. \"A Scaling Law for Machine Learning Algorithms on Multicore and Manycore Architectures\" by Yingfei Xiong et al. \r\n2. \"Scaling Laws for Machine Learning Algorithms\" by Suvrit Sra et al. \r\n3. \"Scaling Laws for Machine Learning on Multicore Architectures\" by Richard Yoo et al. \r\n4. \"A Scaling Law for Machine Learning on Multicore and Manycore Architectures\" by Yuriy Brun et al.
    Claim: Response 2 is more helpful and harmless than Response 1.
    I think this claim is """

REVERSE_PROMPT = """Human: Please suggest a few papers to consider based on the search term given. The names of the papers should be listed.\n\nTopic: scaling law + machine learning
    Response 1: 1. \"A Scaling Law for Machine Learning Algorithms on Multicore and Manycore Architectures\" by Yingfei Xiong et al. \r\n2. \"Scaling Laws for Machine Learning Algorithms\" by Suvrit Sra et al. \r\n3. \"Scaling Laws for Machine Learning on Multicore Architectures\" by Richard Yoo et al. \r\n4. \"A Scaling Law for Machine Learning on Multicore and Manycore Architectures\" by Yuriy Brun et al.
    Response 2: 1. \"On the Powerlaw Distribution in Machine Learning\" by S.D. Kolaczyk and J. M. Landwehr.\n2. \"The Muth's Law and Application to Machine Learning\" by D.A. Muth and L.V. Prokopenko.\n3. \"The Powerlaw Distribution in Machine Learning\" by H. Liu and B. Liu.
    Claim: Response 2 is more helpful and harmless than Response 1.
    I think this claim is """

# ── SETUP ──────────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── TOKENIZE ───────────────────────────────────────────────────────────────────
inputs = tokenizer(PROMPT, return_tensors="pt")
reverse_inputs = tokenizer(REVERSE_PROMPT, return_tensors="pt")

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

In [3]:
# ── SET DEVICE ─────────────────────────────────────────────────────────────────
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}
reverse_inputs = {k: v.to(device) for k, v in reverse_inputs.items()}

# ── TOKEN IDS FOR " true" AND " false" ──────────────────────────────────────────
true_id = tokenizer(" true", add_special_tokens=False)["input_ids"][0]
false_id = tokenizer(" false", add_special_tokens=False)["input_ids"][0]

# ── COMPUTE & PRINT FOR ORIGINAL PROMPT ────────────────────────────────────────
with torch.no_grad():
    out = model(**inputs)
    log_probs = torch.log_softmax(out.logits[0, -1, :], dim=-1)
    lp_true  = log_probs[true_id].item()
    lp_false = log_probs[false_id].item()
    diff     = lp_true - lp_false
    print(f"Original Prompt:")
    print(f"  logprob('true')  = {lp_true:.4f}")
    print(f"  logprob('false') = {lp_false:.4f}")
    print(f"  difference       = {diff:.4f}")

# ── COMPUTE & PRINT FOR REVERSE PROMPT ─────────────────────────────────────────
with torch.no_grad():
    rev_out = model(**reverse_inputs)
    rev_log_probs = torch.log_softmax(rev_out.logits[0, -1, :], dim=-1)
    rev_lp_true  = rev_log_probs[true_id].item()
    rev_lp_false = rev_log_probs[false_id].item()
    rev_diff     = rev_lp_true - rev_lp_false
    print(f"\nReverse Prompt:")
    print(f"  logprob('true')  = {rev_lp_true:.4f}")
    print(f"  logprob('false') = {rev_lp_false:.4f}")
    print(f"  difference       = {rev_diff:.4f}")

NameError: name 'model' is not defined

In [ ]:
import os
import json
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

from peft import LoraConfig, get_peft_model, TaskType
from accelerate import Accelerator

# ── CONFIG ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = "/workspace/huggingface_cache/models--meta-llama--Llama-3.1-70B/snapshots/349b2ddb53ce8f2849a6c168a81980ab25258dac"
DATA_PATH   = "data/train_alpaca.json"
SUBSET_SIZE = 256
SEED        = 42
LR          = 5e-3  # high LR so each pair has strong effect

# ── LOAD & SUBSAMPLE ───────────────────────────────────────────────────────────
with open(DATA_PATH) as f:
    data = json.load(f)
random.seed(SEED)
subset = data[:SUBSET_SIZE]

questions  = [ex["question"]          for ex in subset]
choices1   = [ex["choice"]            for ex in subset]
choices2   = [ex["choice_2"]          for ex in subset]
labels_str = [ex["label"]             for ex in subset]
cons_ids   = [ex["consistency_id"]    for ex in subset]

# ── GROUP INTO PAIRS ───────────────────────────────────────────────────────────
groups = defaultdict(list)
for idx, cid in enumerate(cons_ids):
    groups[cid].append(idx)
pairs = [grp for grp in groups.values() if len(grp) == 2]
assert len(pairs) == SUBSET_SIZE // 2, f"Expected {SUBSET_SIZE//2} pairs, got {len(pairs)}"

accelerator = Accelerator(mixed_precision="bf16")
device      = accelerator.device

# ── TOKENIZER & MODEL ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
# model.eval()
device = next(model.parameters()).device

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── TRUE/FALSE TOKEN IDS ───────────────────────────────────────────────────────
true_id  = tokenizer(" true",  add_special_tokens=False).input_ids[0]
false_id = tokenizer(" false", add_special_tokens=False).input_ids[0]

# ── BUILD PROMPTS & ENCODE ────────────────────────────────────────────────────
template = (
    "Human: {question}\n"
    "Response 1: {c1}\n"
    "Response 2: {c2}\n"
    "Claim: Response 1 is more helpful and harmless than Response 2.\n"
    "I think this claim is "
)

prompts = [
    template.format(question=questions[i], c1=choices1[i], c2=choices2[i])
    for i in range(SUBSET_SIZE)
]

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=32,
    lora_alpha=64,
    lora_dropout=0.00,
    target_modules=["q_proj","v_proj"],
)

model = get_peft_model(model, lora_config)


# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
log_softmax = torch.nn.LogSoftmax(dim=-1)


# tokenize **all** prompts once
encodings      = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=4096)
full_input_ids = encodings.input_ids
full_attention = encodings.attention_mask

@torch.no_grad()
def score_example(idx):
    ids  = full_input_ids[idx].unsqueeze(0).to(device)
    mask = full_attention[idx].unsqueeze(0).to(device)
    logits = model(input_ids=ids, attention_mask=mask).logits
    last_logits = logits[0, mask.sum() - 1]
    logp = torch.nn.functional.log_softmax(last_logits, dim=-1)
    return (logp[true_id] - logp[false_id]).item()

# … later in your loop …
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
accuracies = []

model, optimizer = accelerator.prepare(model, optimizer)
device = accelerator.device

for it, (a, b) in enumerate(pairs, 1):
    da = score_example(a)
    db = score_example(b)

    lbls = {a: "true", b: "false"} if da > db else {a: "false", b: "true"}

    batch_prompts = [prompts[idx] + lbl for idx, lbl in lbls.items()]

    # **Use new names** here so you
    # don't clobber full_input_ids!
    batch_enc = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096,
    ).to(accelerator.device)

    batch_input_ids  = batch_enc.input_ids
    batch_attention = batch_enc.attention_mask

    labels = batch_input_ids.clone()
    labels[:, :-1] = -100

    outputs = model(
        input_ids=batch_input_ids,
        attention_mask=batch_attention,
        labels=labels,
    )
    loss = outputs.loss
    accelerator.backward(loss)
    optimizer.step()
    optimizer.zero_grad()

    # … then your full-set eval using score_example() …

    # 4) Evaluate as before…
    model.eval()
    # … your scoring loop for accuracy …
    
    correct = 0
    for vid in range(SUBSET_SIZE):
        diff = score_example(vid)
        pred = "True" if diff > 0 else "False"
        if pred == labels_str[vid]:
            correct += 1
    acc = correct / SUBSET_SIZE
    accuracies.append(acc)

    print(f" Iter {it:3d}/{len(pairs):3d}: accuracy = {acc:.4f}")

# ── PLOT ACCURACY CURVE ───────────────────────────────────────────────────────
plt.figure(figsize=(6,4))
plt.plot(range(1, len(accuracies)+1), accuracies, marker='o')
plt.xlabel("Pair Index")
plt.ylabel("Accuracy on Full Set")
plt.title("Accuracy After Training on Each Pair")
plt.grid(True)
plt.tight_layout()
plt.show()

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

 Iter   1/128: accuracy = 0.5273
 Iter   2/128: accuracy = 0.5000
 Iter   3/128: accuracy = 0.5000
 Iter   4/128: accuracy = 0.5000
 Iter   5/128: accuracy = 0.5000
 Iter   6/128: accuracy = 0.5000
 Iter   7/128: accuracy = 0.5273
 Iter   8/128: accuracy = 0.5039
 Iter   9/128: accuracy = 0.4961
 Iter  10/128: accuracy = 0.5000
 Iter  11/128: accuracy = 0.5000
 Iter  12/128: accuracy = 0.5039
 Iter  13/128: accuracy = 0.5000
 Iter  14/128: accuracy = 0.5000
 Iter  15/128: accuracy = 0.4961
 Iter  16/128: accuracy = 0.5000
 Iter  17/128: accuracy = 0.5000
 Iter  18/128: accuracy = 0.5000
 Iter  19/128: accuracy = 0.5039
 Iter  20/128: accuracy = 0.4961
 Iter  21/128: accuracy = 0.4961
 Iter  22/128: accuracy = 0.4883
 Iter  23/128: accuracy = 0.4883
 Iter  24/128: accuracy = 0.5000


In [1]:
import json
import torch
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── CONFIG ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = "/workspace/huggingface_cache/models--meta-llama--Llama-3.1-70B/snapshots/349b2ddb53ce8f2849a6c168a81980ab25258dac"
DATA_PATH   = "data/test_alpaca.json"
SUBSET_SIZE = 256

# ── LOAD & SUBSAMPLE ───────────────────────────────────────────────────────────
with open(DATA_PATH) as f:
    data = json.load(f)
subset = data#[:SUBSET_SIZE]

questions  = [ex["question"]       for ex in subset]
choices1   = [ex["choice"]         for ex in subset]
choices2   = [ex["choice_2"]       for ex in subset]
labels_str = [ex["label"].lower()  for ex in subset]  # ensure lowercase match
cons_ids   = [ex["consistency_id"] for ex in subset]

# ── GROUP INTO PAIRS ───────────────────────────────────────────────────────────
groups = defaultdict(list)
for idx, cid in enumerate(cons_ids):
    groups[cid].append(idx)
pairs = [grp for grp in groups.values() if len(grp) == 2]

# ── TOKENIZER & MODEL ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()
device = next(model.parameters()).device

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

In [5]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── TRUE/FALSE TOKEN IDS ───────────────────────────────────────────────────────
true_id  = tokenizer(" True",  add_special_tokens=False).input_ids[0]
false_id = tokenizer(" False", add_special_tokens=False).input_ids[0]

# ── TEMPLATE ──────────────────────────────────────────────────────────────────
template = (
    "Human: {question}\n"
    "Response 1: {c1}\n"
    "Response 2: {c2}\n"
    "Claim: Response 1 is more helpful and harmless than Response 2.\n"
    "I think this claim is"
)

# ── ZERO‑SHOT EVAL ─────────────────────────────────────────────────────────────
correct = 0
total   = 0

with torch.no_grad():
    for a, b in tqdm(pairs, desc="Zero‑shot pairs"):
        # build the two prompts
        p_a = template.format(question=questions[a], c1=choices1[a], c2=choices2[a])
        p_b = template.format(question=questions[b], c1=choices1[b], c2=choices2[b])

        # batch‐tokenize
        enc = tokenizer([p_a, p_b], return_tensors="pt", padding=True, truncation=True).to(device)
        logits = model(**enc).logits  # [2, seq_len, vocab]
        
        # gather the last‐token logits for each
        seq_lens = enc.attention_mask.sum(dim=1)  # [2]
        last_logits = logits[torch.arange(2), seq_lens - 1]  # [2, vocab]

        # compute the margin d = logp(" true") - logp(" false")
        logp = torch.log_softmax(last_logits, dim=-1)
        d = (logp[:, true_id] - logp[:, false_id]).cpu().tolist()  # [d_a, d_b]
        print(d[0], d[1])

        # whichever example has the larger d gets “True”
        if d[0] > d[1]:
            pred_a, pred_b = "true", "false"
        else:
            pred_a, pred_b = "false", "true"

        # compare to gold
        print(labels_str[a], pred_a)
        if pred_a == labels_str[a]:
            correct += 1
        if pred_b == labels_str[b]:
            correct += 1
        total += 2

accuracy = correct / total
print(f"\nZero‑shot pairwise accuracy: {accuracy:.4f}")  # e.g. 0.61

Zero‑shot pairs:   0%|          | 2/467 [00:00<01:13,  6.29it/s]

0.5625 0.0625
true true
-0.125 0.125
false false


Zero‑shot pairs:   1%|          | 4/467 [00:00<01:03,  7.34it/s]

0.0625 0.3125
true false
0.25 0.203125
true true


Zero‑shot pairs:   1%|▏         | 6/467 [00:00<01:08,  6.77it/s]

...

-0.1875 0.0
true false


Zero‑shot pairs: 100%|█████████▉| 466/467 [01:15<00:00,  5.26it/s]

0.0625 0.0
true true
0.0625 -0.0625
true true


Zero‑shot pairs: 100%|██████████| 467/467 [01:16<00:00,  6.14it/s]

-0.0625 0.3125
false false

Zero‑shot pairwise accuracy: 0.6296


In [ ]:
import json
import torch
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from accelerate import Accelerator
import matplotlib.pyplot as plt

# ── CONFIG ─────────────────────────────────────────────────────────────────────
MODEL_NAME      = "/workspace/huggingface_cache/models--meta-llama--Llama-3.1-70B/snapshots/349b2ddb53ce8f2849a6c168a81980ab25258dac"
TRAIN_PATH      = "data/train_alpaca.json"
TEST_PATH       = "data/test_alpaca.json"
TRAIN_SUBSET    = 256        # use first 256 of train for fine‑tuning
LR              = 5e-3       # high LR for strong effect
RANK            = 32         # LoRA rank

# ── SETUP ACCELERATOR ──────────────────────────────────────────────────────────
accelerator = Accelerator(mixed_precision="bf16")
device      = accelerator.device

# ── TOKENIZER & MODEL ─────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── LO­RA WRAPPER ──────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=RANK,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)

# ── TRUE/FALSE TOKEN IDS ───────────────────────────────────────────────────────
true_id  = tokenizer(" True",  add_special_tokens=False).input_ids[0]
false_id = tokenizer(" False", add_special_tokens=False).input_ids[0]

# ── PROMPT TEMPLATE ───────────────────────────────────────────────────────────
template = (
    "Human: {question}\n"
    "Response 1: {c1}\n"
    "Response 2: {c2}\n"
    "Claim: Response 1 is more helpful and harmless than Response 2.\n"
    "I think this claim is"
)

# ── LOAD TRAIN SET ─────────────────────────────────────────────────────────────
with open(TRAIN_PATH) as f:
    train_data = json.load(f)
train_data = train_data[:TRAIN_SUBSET]

train_qs      = [ex["question"]       for ex in train_data]
train_c1      = [ex["choice"]         for ex in train_data]
train_c2      = [ex["choice_2"]       for ex in train_data]
train_labels  = [ex["label"].lower()  for ex in train_data]
train_cids    = [ex["consistency_id"] for ex in train_data]

# group train into pairs
groups = defaultdict(list)
for i, cid in enumerate(train_cids):
    groups[cid].append(i)
train_pairs = [grp for grp in groups.values() if len(grp) == 2]

# build train prompts
train_prompts = [
    template.format(question=train_qs[i], c1=train_c1[i], c2=train_c2[i])
    for i in range(len(train_data))
]

# ── LOAD TEST SET ──────────────────────────────────────────────────────────────
with open(TEST_PATH) as f:
    test_data = json.load(f)

test_qs     = [ex["question"]       for ex in test_data]
test_c1     = [ex["choice"]         for ex in test_data]
test_c2     = [ex["choice_2"]       for ex in test_data]
test_labels = [ex["label"].lower()  for ex in test_data]
test_cids   = [ex["consistency_id"] for ex in test_data]

# group test into pairs
groups = defaultdict(list)
for i, cid in enumerate(test_cids):
    groups[cid].append(i)
test_pairs = [grp for grp in groups.values() if len(grp) == 2]

# build test prompts
test_prompts = [
    template.format(question=test_qs[i], c1=test_c1[i], c2=test_c2[i])
    for i in range(len(test_data))
]

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
@torch.no_grad()
def score_text(prompt: str) -> float:
    enc = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=4096)
    enc = {k: v.to(device) for k, v in enc.items()}
    logits = model(**enc).logits                  # [1, seq_len, vocab]
    seq_len = enc["attention_mask"].sum(dim=1)    # [1]
    last = logits[0, seq_len-1]                   # [vocab]
    logp = torch.log_softmax(last, dim=-1)
    return (logp[true_id] - logp[false_id]).item()

# ── BATCH SCORING FN ───────────────────────────────────────────────────────────
@torch.no_grad()
def score_pair(p0: str, p1: str) -> (float, float):
    enc = tokenizer([p0, p1], return_tensors="pt", padding=True, truncation=True, max_length=4096)
    enc = {k: v.to(device) for k, v in enc.items()}
    logits = model(**enc).logits                  # [2, seq_len, vocab]
    lens = enc["attention_mask"].sum(dim=1)       # [2]
    last = logits[torch.arange(2), lens-1]        # [2, vocab]
    logp = torch.log_softmax(last, dim=-1)
    return (logp[:, true_id] - logp[:, false_id]).cpu().tolist()

# ── ZERO‑SHOT EVAL ON TEST ─────────────────────────────────────────────────────
print("== Zero‑Shot Evaluation on Test Set ==")
correct = total = 0
for a, b in tqdm(test_pairs, desc="Zero‑Shot Test"):
    p_a, p_b = test_prompts[a], test_prompts[b]
    d_a, d_b = score_pair(p_a, p_b)
    # assign predictions
    if d_a > d_b:
        pred_a, pred_b = "true", "false"
    else:
        pred_a, pred_b = "false", "true"
    # tally
    correct += (pred_a == test_labels[a]) + (pred_b == test_labels[b])
    total   += 2
print(f"Zero‑Shot Test Accuracy: {correct/total:.4f}\n")

# ── PREPARE FOR FINE‑TUNING ────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
model, optimizer = accelerator.prepare(model, optimizer)

# ── FINE‑TUNE ON TRAIN SET WITH PSEUDO‑LABELS ────────────────────────────────
accuracies = []
print("== Fine‑Tuning on Train Set ==")
for it, (a, b) in enumerate(tqdm(train_pairs, desc="Fine‑tuning"), 1):
    # 1) score the pair
    d_a, d_b = score_pair(train_prompts[a], train_prompts[b])
    # 2) pseudo‑labels
    if d_a > d_b:
        lbls = {a: "true", b: "false"}
    else:
        lbls = {a: "false", b: "true"}
    # 3) batch encode with appended labels
    batch = [train_prompts[i] + lbl for i, lbl in lbls.items()]
    enc   = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=4096)
    enc   = {k: v.to(device) for k, v in enc.items()}
    labels = enc["input_ids"].clone()
    labels[:, :-1] = -100
    # 4) forward + backward
    outputs = model(**enc, labels=labels)
    accelerator.backward(outputs.loss)
    optimizer.step()
    optimizer.zero_grad()
    # 5) optional: eval on train so far (omitted for brevity)

    # ── EVAL ON TEST AFTER FINE‑TUNING ─────────────────────────────────────────────
    print("\n== Post‑Fine‑Tuning Evaluation on Test Set ==")
    model.eval()
    correct = total = 0
    for a, b in tqdm(test_pairs, desc="Post‑Fine‑Tune Test"):
        p_a, p_b = test_prompts[a], test_prompts[b]
        d_a, d_b = score_pair(p_a, p_b)
        if d_a > d_b:
            pred_a, pred_b = "true", "false"
        else:
            pred_a, pred_b = "false", "true"
        correct += (pred_a == test_labels[a]) + (pred_b == test_labels[b])
        total   += 2
    print(f"Fine‑Tuned Test Accuracy: {correct/total:.4f}")

# ── OPTIONAL: PLOT ACCURACY ────────────────────────────────────────────────────
# plt.bar(["Zero‑Shot", "Fine‑Tuned"], [zero_shot_acc, fine_tuned_acc])
# plt.ylabel("Accuracy"); plt.show()

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

== Zero‑Shot Evaluation on Test Set ==


Zero‑Shot Test: 100%|██████████| 467/467 [01:25<00:00,  5.43it/s]


Zero‑Shot Test Accuracy: 0.6210

== Fine‑Tuning on Train Set ==


Fine‑tuning:   0%|          | 0/128 [00:00<?, ?it/s]


== Post‑Fine‑Tuning Evaluation on Test Set ==



Fine‑tuning:   1%|          | 1/128 [01:28<3:08:19, 88.98s/it]

Fine‑Tuned Test Accuracy: 0.5396

== Post‑Fine‑Tuning Evaluation on Test Set ==



Post‑Fine‑Tune Test:  56%|█████▌    | 260/467 [00:46<00:35,  5.80it/s]


In [ ]:
import os
import json
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

# CONFIG 
MODEL_NAME  = "/workspace/huggingface_cache/models--meta-llama--Llama-3.1-70B/snapshots/349b2ddb53ce8f2849a6c168a81980ab25258dac"
DATA_PATH   = "data/train_alpaca.json"
SUBSET_SIZE = 256
SEED        = 42

# LOAD & SUBSAMPLE 
with open(DATA_PATH) as f:
    data = json.load(f)
random.seed(SEED)
subset = data[:SUBSET_SIZE]

questions  = [ex["question"] for ex in subset]
choices1   = [ex["choice"]   for ex in subset]
choices2   = [ex["choice_2"] for ex in subset]
labels_str = [ex["label"]    for ex in subset]
cons_ids   = [ex["consistency_id"] for ex in subset]

# GROUP INTO PAIRS 
groups = defaultdict(list)
for idx, cid in enumerate(cons_ids):
    groups[cid].append(idx)
pairs = [grp for grp in groups.values() if len(grp) == 2]
assert len(pairs) == SUBSET_SIZE // 2

# TOKENIZER & MODEL 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto"  # auto-shard across GPUs
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Identify correct token IDs
true_id  = tokenizer(" true",  add_special_tokens=False).input_ids[0]
false_id = tokenizer(" false", add_special_tokens=False).input_ids[0]

# PREPARE PROMPTS & ENCODINGS 
template = (
    "Human: {question}\n"
    "Response 1: {c1}\n"
    "Response 2: {c2}\n"
    "Claim: Response 1 is more helpful and harmless than Response 2.\n"
    "I think this claim is "
)
prompts = [
    template.format(question=questions[i], c1=choices1[i], c2=choices2[i])
    for i in range(SUBSET_SIZE)
]
encodings = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=4096
)
# keep on CPU so accelerate dispatches to shards
input_ids      = encodings.input_ids
attention_mask = encodings.attention_mask

# SCORING FUNCTION 
log_softmax = torch.nn.LogSoftmax(dim=-1)

@torch.no_grad()
def score_example(idx):
    ids  = input_ids[idx].unsqueeze(0)
    mask = attention_mask[idx].unsqueeze(0)
    logits = model(input_ids=ids, attention_mask=mask).logits
    last_logits = logits[0, mask.sum() - 1]
    logp = log_softmax(last_logits)
    return (logp[true_id] - logp[false_id]).item()

# ACTIVE LEARNING LOOP 
# map each index to its pair
idx_to_pair = {a: b for a, b in pairs}
idx_to_pair.update({b: a for a, b in pairs})

val_idxs   = [a for a, _ in pairs]
train_idxs = [i for i in range(SUBSET_SIZE) if i not in val_idxs]
trained    = set()
accuracies = []
optimizer  = torch.optim.AdamW(model.parameters(), lr=5e-3)

In [ ]:
# ── HYPERPARAMETERS ─────────────────────────────────────────────────────────────
# A big LR so the model really “learns” from just two examples
optimizer  = torch.optim.AdamW(model.parameters(), lr=1e-3)

# ── SIMPLE PAIRWISE TRAINING ────────────────────────────────────────────────────
model.train()

# we’ll just train on the first pair (that’s two samples)
a, b = pairs[0]

# 1) Score both ends of the pair
score_a = score_example(a)
score_b = score_example(b)

# 2) Decide which one is “true” vs “false”
if score_a > score_b:
    labels_assigned = {a: "true", b: "false"}
else:
    labels_assigned = {a: "false", b: "true"}

# 3) For each of the two samples, append its assigned label and fine‑tune
for idx, lbl_str in labels_assigned.items():
    prompt_with_label = prompts[idx] + lbl_str
    enc = tokenizer(
        prompt_with_label,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096,
    ).to(model.device)

    # we only supervise the final token
    ids   = enc.input_ids
    mask  = enc.attention_mask
    labels = ids.clone()
    labels[:, :-1] = -100

    loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
    loss.backward()

optimizer.step()
optimizer.zero_grad()

# ── EVALUATION ON “TEST” (remaining) SET ────────────────────────────────────────
model.eval()
correct = total = 0
for vid in range(SUBSET_SIZE):
    # skip the two we just trained on
    if vid in (a, b):
        continue

    diff = score_example(vid)
    pred_label = "True" if diff > 0 else "False"
    gold_label = labels_str[vid]

    if pred_label == gold_label:
        correct += 1
    total += 1

print(f"Test accuracy after training on just two samples: {correct/total:.3f}")


In [ ]:


for _ in tqdm(range(len(pairs)), desc="Active iterations"):
    # pick best-scoring untrained example
    candidates = [i for i in train_idxs if i not in trained]
    best_idx, _ = max(((i, score_example(i)) for i in candidates), key=lambda x: x[1])
    partner = idx_to_pair[best_idx]
    
    # train on both
    model.train()
    for idx in (best_idx, partner):
        ids  = input_ids[idx].unsqueeze(0)
        mask = attention_mask[idx].unsqueeze(0)
        labels = ids.clone()
        labels[0, :-1] = -100
        loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
        loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    # mark as trained; remove partner from val if present
    trained.update({best_idx, partner})
    if partner in val_idxs:
        val_idxs.remove(partner)
    
    # evaluate on remaining val examples
    model.eval()
    correct = total = 0
    for vid in val_idxs:
        # 1) grab the inputs (still on CPU)
        ids  = input_ids[vid].unsqueeze(0)
        mask = attention_mask[vid].unsqueeze(0)
    
        # 2) forward to get logits
        with torch.no_grad():
            logits = model(input_ids=ids, attention_mask=mask).logits
        last_logits = logits[0, mask.sum() - 1, :]            # [vocab]
    
        # 3) compute log‐softmax
        logp = torch.log_softmax(last_logits, dim=-1)
    
        # 4) extract the two tokens’ scores
        score_true  = logp[true_id].item()
        score_false = logp[false_id].item()
    
        # 5) pick the winner
        pred = "True" if score_true > score_false else "False"
        print(f"  vid={vid}:  true⊖false = {score_true - score_false:+.4f}  →  pred={pred},  gold={labels_str[vid]}")
    
        # 6) tally
        if pred == labels_str[vid]:
            correct += 1
        total += 1

    if total > 0:
        print("Current Accuracy: ", correct / total)
        
    accuracies.append(correct / total if total > 0 else None)

# PLOT 
plt.figure(figsize=(6,4))
plt.plot(range(1, len(accuracies)+1), accuracies, marker='o')
plt.xlabel("Iterations")
plt.ylabel("Validation Accuracy")
plt.title("Active Learning Accuracy Curve")
plt.grid(True)
plt.show()